# Chapter 16 — GPU Attention

> Course: **llm.c — Zero to Hero**, Chapter 16 of ~20.
> Builds on Chapter 6 (CPU attention) and Chapters 12-13 (warp+block reductions).

Attention is the longest function in `train_gpt2.c` and arguably the longest section in `llmc/`. On GPU it gets even more involved because:

1. The Q·K matmul is huge — `(B, NH, T, T)` scores matrix that scales **quadratically** with `T`.
2. The softmax is per-row, autoregressive (causal mask), and numerically delicate.
3. Memory layout matters — to use cuBLAS for QKᵀ, we need Q, K, V **separated** and transposed, not packed.
4. There's a *much* faster algorithm — **Flash Attention** — available via cuDNN that we should use when possible.

This chapter is mostly a tour of `llmc/attention.cuh` and `llmc/cudnn_att.h`. We'll write a small softmax-with-mask kernel to ground the discussion, but the production attention is too large to teach line by line.

### Learning objectives

By the end of this chapter you will:

- Explain why attention requires `permute_kernel`: the QKV layout swap.
- Read `softmax_forward_kernel5` and explain its 3-pass + warp-shuffle structure.
- Articulate Flash Attention's central trick: tiling + online softmax = no `(B, NH, T, T)` materialization.
- Choose between hand-rolled and cuDNN Flash Attention based on `T` and the GPU.


## 1. Concept — Why GPU Attention Is Different

The CPU attention from Chapter 6 walked one `(b, t, h)` tuple at a time and looped over `t2`. On GPU we have to think in batched matrix multiplies:

```
Q  : (B, NH, T,    hs)      (separated from packed QKV)
K  : (B, NH, T,    hs)
V  : (B, NH, T,    hs)
S  = Q @ K^T  : (B, NH, T, T)        ← cuBLAS
S  = S * scale + causal_mask         ← elementwise + masked softmax
A  = softmax(S, dim=-1) : (B, NH, T, T)
out = A @ V  : (B, NH, T, hs)        ← cuBLAS
permute back to (B, T, NH*hs) = (B, T, C)
```

That's two cuBLAS batched-matmul calls plus the masked softmax in between. Three challenges:

1. **`(B, NH, T, T)` is huge.** For GPT-2 small at `T=1024`: `4 × 12 × 1024 × 1024` floats = ~200 MB of scores. Storing this is a real cost.
2. **Memory layout.** `llm.c` keeps Q, K, V *packed* in `(B, T, 3C)` for cache-friendly QKV linear projections. cuBLAS needs them *separated and transposed* to `(B, NH, T, hs)`. So we need a `permute_kernel` to shuffle data into that layout, and another `unpermute_kernel` after.
3. **Causal masking.** The softmax must zero out future positions (`t2 > t`). One option: write `-inf` into the upper triangle of `S`. Another: build a kernel that *just doesn't read* those positions.


## 2. Walkthrough — `permute_kernel`

From [`llmc/attention.cuh`](llmc/attention.cuh) (paraphrased):

```cpp
__global__ void permute_kernel(floatX* q, floatX* k, floatX* v,
                               const floatX* inp, int B, int N, int NH, int d) {
    // inp is (B, N, 3, NH, d)  -- packed Q, K, V interleaved
    // produce q, k, v all (B, NH, N, d)
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx >= B*NH*N*d) return;

    int b   = idx / (NH * N * d);
    int rest = idx % (NH * N * d);
    int nh  = rest / (N * d);
    int n   = (rest / d) % N;
    int dd  = rest % d;

    int inp_idx = b*N*3*NH*d + n*3*NH*d + 0*NH*d + nh*d + dd;     // Q region
    q[idx] = inp[inp_idx];
    k[idx] = inp[inp_idx + NH*d];          // +NH*d to land in K region
    v[idx] = inp[inp_idx + 2*NH*d];        // +2*NH*d for V
}
```

What's happening: each thread owns one element of the output `(B, NH, N, d)` view. It computes which `(b, nh, n, d)` it represents, then **looks up** the corresponding bytes in the packed `(B, N, 3, NH, d)` input. The output Q, K, V are now in the layout cuBLAS wants.

This is a **reshape with axis swap** in the abstract — and on GPU it has to be an explicit kernel because we can't just adjust strides like in PyTorch (the bytes literally need to move to be cuBLAS-friendly). Memory pattern: reads are non-coalesced (jumping across the packed dim), writes are coalesced. This is a deliberate tradeoff.


## 3. Walkthrough — `softmax_forward_kernel5`

From [`llmc/attention.cuh`](llmc/attention.cuh) — the masked softmax over the `(T, T)` last-two-dims of the scores tensor:

```cpp
__global__ void softmax_forward_kernel5(floatX* out, float scale, const floatX* inp,
                                        int N, int T) {
    // each warp handles one row of the (N, T) tensor
    extern __shared__ float shared[];
    int idx = blockIdx.x * (blockDim.x / 32) + (threadIdx.x / 32);   // row index
    int tid = threadIdx.x % 32;                                       // lane in warp
    if (idx >= N) return;

    int own_pos = idx % T;       // current query position; mask = (col <= own_pos)
    const floatX* x = inp + idx * T;
    floatX* y       = out + idx * T;

    // pass 1: max over the row, masked
    float maxval = -INFINITY;
    for (int i = tid; i <= own_pos; i += 32) {
        float v = (float)x[i] * scale;
        maxval = fmaxf(maxval, v);
    }
    maxval = warpReduceMax(maxval);   // lane 0 holds the row max
    maxval = __shfl_sync(0xffffffff, maxval, 0);

    // pass 2: sum of exp(x - max), masked
    float sum = 0.0f;
    for (int i = tid; i <= own_pos; i += 32) sum += expf((float)x[i] * scale - maxval);
    sum = warpReduceSum(sum);
    sum = __shfl_sync(0xffffffff, sum, 0);
    float invsum = 1.0f / sum;

    // pass 3: write out
    for (int i = tid; i < T; i += 32) {
        float v = (i <= own_pos) ? expf((float)x[i] * scale - maxval) * invsum : 0.0f;
        y[i] = (floatX)v;
    }
}
```

What's interesting:

1. **One warp per row** — `T` is at most 1024 in `llm.c`, so 32 threads doing strided loads (`for i = tid; i <= own_pos; i += 32`) covers any row in 32 iterations.
2. **The causal mask is implicit** — passes 1 and 2 only iterate `i <= own_pos`. Pass 3 explicitly writes 0 for `i > own_pos` so the `att @ V` matmul doesn't see garbage there.
3. **Warp shuffles for max + sum** — same pattern as Chapter 12. No shared memory needed for the reduction (only ~32 entries × 4 warps per block fit comfortably in registers + warp-private).

This is the kind of kernel where every line matters and a wrong stride costs you 30% performance.


## 4. Demo — Masked Softmax in 30 Lines

Let's write a much simpler version of `softmax_forward_kernel5` and verify it.


In [ ]:
!mkdir -p course/ch16_build


In [ ]:
%%writefile course/ch16_build/masked_softmax.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

__device__ float warpReduceMax(float v) {
    for (int o = 16; o > 0; o /= 2) v = fmaxf(v, __shfl_down_sync(0xffffffff, v, o));
    return v;
}
__device__ float warpReduceSum(float v) {
    for (int o = 16; o > 0; o /= 2) v += __shfl_down_sync(0xffffffff, v, o);
    return v;
}

// One warp per row of an (N, T) tensor. Causal: only positions <= row%T contribute.
// We treat each (b, nh, t) as one of N rows; T is the column count.
__global__ void masked_softmax_kernel(float* out, const float* scores, int N, int T, int rows_per_block) {
    int row = blockIdx.x * rows_per_block + (threadIdx.x / 32);
    int tid = threadIdx.x % 32;
    if (row >= N) return;
    int own_pos = row % T;

    const float* x = scores + row * T;
    float* y       = out    + row * T;

    float maxval = -1e30f;
    for (int i = tid; i <= own_pos; i += 32) maxval = fmaxf(maxval, x[i]);
    maxval = warpReduceMax(maxval);
    maxval = __shfl_sync(0xffffffff, maxval, 0);

    float sum = 0.0f;
    for (int i = tid; i <= own_pos; i += 32) sum += expf(x[i] - maxval);
    sum = warpReduceSum(sum);
    sum = __shfl_sync(0xffffffff, sum, 0);
    float invsum = 1.0f / sum;

    for (int i = tid; i < T; i += 32) {
        float v = (i <= own_pos) ? expf(x[i] - maxval) * invsum : 0.0f;
        y[i] = v;
    }
}

int main(void) {
    int B = 2, NH = 4, T = 64;
    int N = B * NH * T;       // 512 rows
    float* h_scores = (float*) malloc(N*T*4);
    float* h_gpu    = (float*) malloc(N*T*4);
    float* h_cpu    = (float*) malloc(N*T*4);
    for (int i = 0; i < N*T; i++) h_scores[i] = (float)((i*13)%97) / 50.0f - 0.5f;

    // CPU reference (causal masked softmax per row)
    for (int r = 0; r < N; r++) {
        int own = r % T;
        float m = -1e30f;
        for (int i = 0; i <= own; i++) m = fmaxf(m, h_scores[r*T + i]);
        float s = 0; for (int i = 0; i <= own; i++) s += expf(h_scores[r*T + i] - m);
        for (int i = 0; i < T; i++)
            h_cpu[r*T + i] = (i <= own) ? expf(h_scores[r*T + i] - m) / s : 0.0f;
    }

    float *d_scores, *d_out;
    cudaMalloc(&d_scores, N*T*4); cudaMalloc(&d_out, N*T*4);
    cudaMemcpy(d_scores, h_scores, N*T*4, cudaMemcpyHostToDevice);

    int rows_per_block = 4;
    int block = rows_per_block * 32;        // 4 warps
    int grid  = (N + rows_per_block - 1) / rows_per_block;
    masked_softmax_kernel<<<grid, block>>>(d_out, d_scores, N, T, rows_per_block);
    cudaMemcpy(h_gpu, d_out, N*T*4, cudaMemcpyDeviceToHost);

    float maxerr = 0;
    for (int i = 0; i < N*T; i++) {
        float e = fabsf(h_gpu[i] - h_cpu[i]);
        if (e > maxerr) maxerr = e;
    }
    printf("Masked softmax max diff: %.2e\n", maxerr);

    // also check rows sum to 1 over the unmasked part
    int row = 100;     // arbitrary
    int own = row % T;
    double sum = 0; for (int i = 0; i <= own; i++) sum += h_gpu[row*T + i];
    int n_zero = 0; for (int i = own+1; i < T; i++) if (h_gpu[row*T+i] == 0) n_zero++;
    printf("row %d (own_pos=%d): sum over unmasked = %.6f, masked entries = %d, zeroes seen = %d\n",
           row, own, sum, T - own - 1, n_zero);

    cudaFree(d_scores); cudaFree(d_out);
    free(h_scores); free(h_gpu); free(h_cpu);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch16_build/masked_softmax course/ch16_build/masked_softmax.cu && ./course/ch16_build/masked_softmax


You should see ~`1e-7` agreement and the masked entries being exactly zero. **Causal masked softmax in 30 lines of CUDA.** This is structurally identical to `softmax_forward_kernel5` in `llm.c`; the production version differs only in `floatX` types, `Packed128` loads, and configurable warps-per-row.


## 5. Flash Attention via cuDNN

The above approach has a major weakness: the `(B, NH, T, T)` scores tensor must be materialized to global memory. For `T = 4096` that's `4 × 12 × 4096 × 4096 × 4` bytes = **3 GB** of intermediate. This becomes the bottleneck.

**Flash Attention** (Dao et al. 2022) gets around it. The key insight: you don't have to store the full attention matrix. You can compute attention in **tiles**, processing one block of `(query_rows, key_cols)` at a time, while *online* updating the per-query softmax max and sum. Over the whole row, you accumulate the output `(att @ V)` without ever writing the full `att` row to memory.

The result:
- **No** `(B, NH, T, T)` materialization — `O(T²)` memory becomes `O(T)` working memory.
- **Substantial speedup** — fewer global memory transactions even for moderate T.
- **Quadratic FLOP cost still** — attention is fundamentally `O(T²)` compute, no way around that.

Writing Flash Attention from scratch is hard. Fortunately, **cuDNN provides it**. `llmc/cudnn_att.h` is a wrapper:

```cpp
// from llmc/cudnn_att.h (paraphrased)
void attention_forward_cudnn(floatX* out, float* stats,        // out shape (B, T, NH, hs)
                             floatX* qkvr,                      // permuted Q,K,V tensor
                             int B, int T, int NH, int C,
                             cudaStream_t stream);
```

`llm.c` builds with `USE_CUDNN=1` to enable this. The win at `T=1024` is about 1.5–3× over the hand-rolled version; at `T=4096+` it's much larger.

For *training*, cuDNN's Flash Attention also provides backward — saving you ~150 lines of attention_backward code that you'd otherwise have to write.


## 6. The Decision Tree

When writing GPT-2-style attention on GPU:

| Setting | Choice |
|---|---|
| `T ≤ 256` (small) | Hand-rolled is fine; cuDNN's launch overhead may hurt |
| `T = 512–1024` (GPT-2 small/medium) | cuDNN starts to win |
| `T ≥ 2048` (long-context) | cuDNN Flash Attention is essential — `O(T²)` memory becomes infeasible |
| Inference, single token | Hand-rolled "decoding" attention (`T_out = 1`) |
| Cutting-edge GPU (H100/Blackwell) | cuDNN exposes hardware Flash Attention units; nothing else compares |

`llm.c`'s default `train_gpt2.cu` builds against cuDNN if it's available — the code is faster and shorter.


## 7. Translation Bridge

| PyTorch | GPU `llm.c` |
|---|---|
| `q, k, v = qkv.chunk(3, dim=-1)` then `.view(...).transpose(...)` | `permute_kernel` actually moves bytes |
| `(q @ k.T) * scale` | cuBLAS batched gemm |
| `F.softmax(... + mask)` | `softmax_forward_kernel5` (warp-per-row + causal mask + warp shuffles) |
| `att @ v` | cuBLAS batched gemm again |
| `F.scaled_dot_product_attention(q, k, v, is_causal=True)` | `cudnn_att.h::attention_forward_cudnn` (Flash Attention) |


## 8. TODO Exercise — Add a Causal Mask to a Plain Softmax

You have a 32-wide row already. Add a causal cutoff: only positions `[0, own_pos]` participate.


In [ ]:
%%writefile course/ch16_build/exercise1.cu
#include <stdio.h>
#include <math.h>
#include <cuda_runtime.h>

__device__ float warpReduceMax(float v) { for (int o=16;o>0;o/=2) v = fmaxf(v, __shfl_down_sync(0xffffffff,v,o)); return v; }
__device__ float warpReduceSum(float v) { for (int o=16;o>0;o/=2) v += __shfl_down_sync(0xffffffff,v,o); return v; }

// One row of length 32. own_pos passed in. Output: causal-masked softmax of inp[0..32).
__global__ void causal_softmax_32(float* out, const float* inp, int own_pos) {
    int t = threadIdx.x;
    // TODO: load inp[t]. Mask: if t > own_pos, treat as -inf for the max+sum.
    float v = inp[t];
    // TODO: maxval = warpReduceMax over t in [0, own_pos]
    // hint: use a sentinel like -1e30f for masked lanes
    // TODO: ev = (t <= own_pos) ? expf(v - maxval) : 0.0f
    // TODO: sum = warpReduceSum, broadcast
    // TODO: out[t] = ev / sum if t <= own_pos else 0
    out[t] = 0.0f;
}

int main(void) {
    float h_inp[32], h_out[32], h_ref[32];
    for (int i = 0; i < 32; i++) h_inp[i] = (float)((i*7) % 13) - 6.0f;
    int own_pos = 17;
    // CPU reference
    float m = -1e30f; for (int i = 0; i <= own_pos; i++) m = fmaxf(m, h_inp[i]);
    float s = 0; for (int i = 0; i <= own_pos; i++) s += expf(h_inp[i] - m);
    for (int i = 0; i < 32; i++) h_ref[i] = (i <= own_pos) ? expf(h_inp[i] - m) / s : 0.0f;

    float *d_inp, *d_out;
    cudaMalloc(&d_inp, 32*4); cudaMalloc(&d_out, 32*4);
    cudaMemcpy(d_inp, h_inp, 32*4, cudaMemcpyHostToDevice);
    causal_softmax_32<<<1, 32>>>(d_out, d_inp, own_pos);
    cudaMemcpy(h_out, d_out, 32*4, cudaMemcpyDeviceToHost);

    float maxerr = 0;
    for (int i = 0; i < 32; i++) { float e = fabsf(h_out[i] - h_ref[i]); if (e > maxerr) maxerr = e; }
    printf("max diff: %.2e %s\n", maxerr, maxerr < 1e-5 ? "PASS" : "FAIL");
    cudaFree(d_inp); cudaFree(d_out);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch16_build/exercise1 course/ch16_build/exercise1.cu && ./course/ch16_build/exercise1


### Solution

In [ ]:
%%writefile course/ch16_build/exercise1_sol.cu
#include <stdio.h>
#include <math.h>
#include <cuda_runtime.h>

__device__ float warpReduceMax(float v) { for (int o=16;o>0;o/=2) v = fmaxf(v, __shfl_down_sync(0xffffffff,v,o)); return v; }
__device__ float warpReduceSum(float v) { for (int o=16;o>0;o/=2) v += __shfl_down_sync(0xffffffff,v,o); return v; }

__global__ void causal_softmax_32(float* out, const float* inp, int own_pos) {
    int t = threadIdx.x;
    float v = inp[t];
    float for_max = (t <= own_pos) ? v : -1e30f;
    float maxval  = warpReduceMax(for_max);
    maxval = __shfl_sync(0xffffffff, maxval, 0);
    float ev = (t <= own_pos) ? expf(v - maxval) : 0.0f;
    float sum = warpReduceSum(ev);
    sum = __shfl_sync(0xffffffff, sum, 0);
    out[t] = (t <= own_pos) ? ev / sum : 0.0f;
}

int main(void) {
    float h_inp[32], h_out[32], h_ref[32];
    for (int i = 0; i < 32; i++) h_inp[i] = (float)((i*7) % 13) - 6.0f;
    int own_pos = 17;
    float m = -1e30f; for (int i = 0; i <= own_pos; i++) m = fmaxf(m, h_inp[i]);
    float s = 0; for (int i = 0; i <= own_pos; i++) s += expf(h_inp[i] - m);
    for (int i = 0; i < 32; i++) h_ref[i] = (i <= own_pos) ? expf(h_inp[i] - m) / s : 0.0f;

    float *d_inp, *d_out;
    cudaMalloc(&d_inp, 32*4); cudaMalloc(&d_out, 32*4);
    cudaMemcpy(d_inp, h_inp, 32*4, cudaMemcpyHostToDevice);
    causal_softmax_32<<<1, 32>>>(d_out, d_inp, own_pos);
    cudaMemcpy(h_out, d_out, 32*4, cudaMemcpyDeviceToHost);
    float maxerr = 0;
    for (int i = 0; i < 32; i++) { float e = fabsf(h_out[i] - h_ref[i]); if (e > maxerr) maxerr = e; }
    printf("max diff: %.2e %s\n", maxerr, maxerr < 1e-5 ? "PASS" : "FAIL");
    cudaFree(d_inp); cudaFree(d_out);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch16_build/exercise1_sol course/ch16_build/exercise1_sol.cu && ./course/ch16_build/exercise1_sol


## Recap

You now know:

- GPU attention has a layout dance: `permute_kernel` separates packed `(B, T, 3C)` into `(B, NH, T, hs)` for cuBLAS.
- `softmax_forward_kernel5` runs **one warp per row**, three passes (max, exp+sum, normalize), with implicit causal mask via `i <= own_pos`.
- **Flash Attention** (cuDNN) avoids materializing the `(B, NH, T, T)` scores tensor — essential for `T ≥ 1024`.
- The general decision: hand-roll for very small T, cuDNN for everything else.

### What's next

**Chapter 17 — Mixed Precision (BF16/FP16).** GPT-2 in float32 is ~500 MB of weights; in bfloat16 it's 250 MB — and tensor cores run BF16 matmuls 2× faster than FP32. We'll learn the **master weights** trick that makes mixed precision stable, and meet the `floatX` typedef that toggles the precision of every layer at compile time.

When you're ready, say **"proceed to Chapter 17"**.
